# GF-MUL Decoder SASCA (정리본)

이 노트북은 **실제 prediction matrix 두 개**를 모두 사용하는 **practical 버전**이다.

- `op1_hw` evidence: prediction matrix 사용
- `out_hw` evidence: prediction matrix 사용

논문과의 대응:
- `gf_mul` 단일 연산 evidence: Section 3
- real `H_eff` 사용: decoder 계수
- Figure 5 `log → +log(h) → mod255 → exp` one-window BP: Section 4
- 출력: 각 byte posterior, top-k 후보, 다음 단계(re-decoding)에 넘길 attack package


In [7]:
import numpy as np
import h5py
from scalib.attacks import FactorGraph, BPState

# ----------------------------
# User config
# ----------------------------
PRED_PATH = "/Users/ijaeyeon/chipwhisperer/jupyter/user/predict_matrix/pred_matrix_gf_mul_F405_500000_pred100000_42_pred42.h5"
SEED = 1234
TOPK = 32

rng = np.random.default_rng(SEED)


In [8]:
# ----------------------------
# Load practical prediction matrices
# ----------------------------
with h5py.File(PRED_PATH, "r") as f:
    op1_hw_pred = f["op1_hw"]["pred_matrix"][:].astype(np.float64)
    op1_hw_true = f["op1_hw"]["true_labels"][:].astype(np.uint8)

    out_hw_pred = f["out_hw"]["pred_matrix"][:].astype(np.float64)
    out_hw_true = f["out_hw"]["true_labels"][:].astype(np.uint8)

print("op1_hw_pred:", op1_hw_pred.shape, op1_hw_pred.dtype)
print("out_hw_pred:", out_hw_pred.shape, out_hw_pred.dtype)


op1_hw_pred: (100000, 9) float64
out_hw_pred: (100000, 9) float64


In [9]:
# ----------------------------
# Basic helpers
# ----------------------------
def build_label_buckets(true_labels: np.ndarray, n_classes: int):
    return [np.where(true_labels == c)[0] for c in range(n_classes)]

op1_hw_buckets = build_label_buckets(op1_hw_true, n_classes=9)
out_hw_buckets = build_label_buckets(out_hw_true, n_classes=9)

def sample_posterior_by_true_label(pred_matrix, buckets, true_label, rng):
    idx_pool = buckets[int(true_label)]
    if len(idx_pool) == 0:
        raise ValueError(f"No samples available for true_label={true_label}")
    idx = rng.choice(idx_pool)
    p = pred_matrix[idx].astype(np.float64)
    s = p.sum()
    if s <= 0:
        raise ValueError("Posterior vector has non-positive sum")
    return p / s

def hw8_scalar(x: int) -> int:
    return bin(x & 0xFF).count("1")

def gf_mul_ref_python(a: int, b: int) -> int:
    # HQC GF(2^8), reduction x^8 + x^4 + x^3 + x^2 + 1  => 0x1D
    a &= 0xFF
    b &= 0xFF
    p = 0
    for _ in range(8):
        if b & 1:
            p ^= a
        hi = a & 0x80
        a = (a << 1) & 0xFF
        if hi:
            a ^= 0x1D
        b >>= 1
    return p & 0xFF

def posterior_rank(post: np.ndarray, true_idx: int) -> int:
    return 1 + int(np.sum(post > post[true_idx]))


In [10]:
import numpy as np

# ============================================================
# Step 2. Real decoder coefficient matrix H_eff
# ============================================================
# Extracted from 2023 reference code: alpha_ij_pow[30][45]
# shape = (n-k, n-1) = (30, 45) for HQC128

H_eff = np.array([[2, 4, 8, 16, 32, 64, 128, 29, 58, 116, 232, 205, 135, 19, 38, 76, 152, 45, 90, 180, 117, 234, 201, 143, 3, 6, 12, 24, 48, 96, 192, 157, 39, 78, 156, 37, 74, 148, 53, 106, 212, 181, 119, 238, 193], [4, 16, 64, 29, 116, 205, 19, 76, 45, 180, 234, 143, 6, 24, 96, 157, 78, 37, 148, 106, 181, 238, 159, 70, 5, 20, 80, 93, 105, 185, 222, 95, 97, 153, 94, 101, 137, 30, 120, 253, 211, 107, 177, 254, 223], [8, 64, 58, 205, 38, 45, 117, 143, 12, 96, 39, 37, 53, 181, 193, 70, 10, 80, 186, 185, 161, 97, 47, 101, 15, 120, 231, 107, 127, 223, 182, 217, 134, 68, 26, 208, 206, 62, 237, 59, 197, 102, 23, 184, 169], [16, 29, 205, 76, 180, 143, 24, 157, 37, 106, 238, 70, 20, 93, 185, 95, 153, 101, 30, 253, 107, 254, 91, 217, 17, 13, 208, 129, 248, 59, 151, 133, 184, 79, 132, 168, 82, 73, 228, 230, 198, 252, 123, 227, 150], [32, 116, 38, 180, 3, 96, 156, 106, 193, 5, 160, 185, 190, 94, 15, 253, 214, 223, 226, 17, 26, 103, 124, 59, 51, 46, 169, 132, 77, 85, 114, 230, 145, 215, 255, 150, 55, 174, 100, 28, 167, 89, 239, 172, 36], [64, 205, 45, 143, 96, 37, 181, 70, 80, 185, 97, 101, 120, 107, 223, 217, 68, 208, 62, 59, 102, 184, 33, 168, 85, 228, 191, 252, 241, 150, 110, 130, 7, 221, 89, 195, 138, 61, 251, 44, 207, 173, 8, 58, 38], [128, 19, 117, 24, 156, 181, 140, 93, 161, 94, 60, 107, 163, 67, 26, 129, 147, 102, 109, 132, 41, 57, 209, 252, 255, 98, 87, 200, 224, 89, 155, 18, 245, 11, 233, 173, 16, 232, 45, 3, 157, 53, 159, 40, 185], [29, 76, 143, 157, 106, 70, 93, 95, 101, 253, 254, 217, 13, 129, 59, 133, 79, 168, 73, 230, 252, 227, 149, 130, 28, 81, 195, 18, 247, 44, 27, 2, 58, 152, 3, 39, 212, 140, 186, 190, 202, 231, 225, 175, 26], [58, 45, 12, 37, 193, 80, 161, 101, 231, 223, 134, 208, 237, 102, 169, 168, 146, 191, 179, 150, 87, 7, 166, 195, 36, 251, 125, 173, 64, 38, 143, 39, 181, 10, 185, 47, 120, 127, 217, 26, 62, 197, 184, 21, 85], [116, 180, 96, 106, 5, 185, 94, 253, 223, 17, 103, 59, 46, 132, 85, 230, 215, 150, 174, 28, 89, 172, 244, 44, 108, 32, 38, 3, 156, 193, 160, 190, 15, 214, 226, 26, 124, 51, 169, 77, 114, 145, 255, 55, 100], [232, 234, 39, 238, 160, 97, 60, 254, 134, 103, 118, 184, 84, 57, 145, 227, 220, 7, 162, 172, 245, 176, 71, 58, 180, 192, 181, 40, 95, 15, 177, 175, 208, 147, 46, 21, 73, 99, 241, 55, 200, 166, 43, 122, 44], [205, 143, 37, 70, 185, 101, 107, 217, 208, 59, 184, 168, 228, 252, 150, 130, 221, 195, 61, 44, 173, 58, 117, 39, 193, 186, 47, 231, 182, 26, 237, 23, 21, 146, 145, 219, 87, 56, 242, 36, 139, 54, 64, 45, 96], [135, 6, 53, 20, 190, 120, 163, 13, 237, 46, 84, 228, 229, 98, 100, 81, 69, 251, 131, 32, 45, 192, 238, 186, 94, 187, 217, 189, 236, 169, 82, 209, 241, 220, 28, 242, 72, 22, 173, 116, 201, 37, 140, 222, 15], [19, 24, 181, 93, 94, 107, 67, 129, 102, 132, 57, 252, 98, 200, 89, 18, 11, 173, 232, 3, 53, 40, 194, 231, 226, 189, 197, 158, 170, 145, 75, 25, 166, 69, 235, 54, 29, 234, 37, 5, 95, 120, 91, 52, 59], [38, 96, 193, 185, 15, 223, 26, 59, 169, 85, 145, 150, 100, 89, 36, 44, 1, 38, 96, 193, 185, 15, 223, 26, 59, 169, 85, 145, 150, 100, 89, 36, 44, 1, 38, 96, 193, 185, 15, 223, 26, 59, 169, 85, 145], [76, 157, 70, 95, 253, 217, 129, 133, 168, 230, 227, 130, 81, 18, 44, 2, 152, 39, 140, 190, 231, 175, 31, 23, 77, 209, 219, 25, 162, 36, 88, 4, 45, 78, 5, 97, 211, 67, 62, 46, 154, 191, 171, 50, 89], [152, 78, 10, 153, 214, 68, 147, 79, 146, 215, 220, 221, 69, 11, 1, 152, 78, 10, 153, 214, 68, 147, 79, 146, 215, 220, 221, 69, 11, 1, 152, 78, 10, 153, 214, 68, 147, 79, 146, 215, 220, 221, 69, 11, 1], [45, 37, 80, 101, 223, 208, 102, 168, 191, 150, 7, 195, 251, 173, 38, 39, 10, 47, 127, 26, 197, 21, 115, 219, 100, 242, 245, 54, 205, 96, 70, 97, 107, 68, 59, 33, 228, 241, 130, 89, 61, 207, 58, 12, 193], [90, 148, 186, 30, 226, 62, 109, 73, 179, 174, 162, 61, 131, 232, 96, 140, 153, 127, 52, 51, 168, 99, 98, 56, 172, 22, 8, 234, 212, 185, 240, 67, 237, 79, 114, 241, 25, 121, 245, 108, 19, 39, 20, 188, 223], [180, 106, 185, 253, 17, 59, 132, 230, 150, 28, 172, 44, 32, 3, 193, 190, 214, 26, 51, 77, 145, 55, 167, 36, 233, 116, 96, 5, 94, 223, 103, 46, 85, 215, 174, 89, 244, 108, 38, 156, 160, 15, 226, 124, 169], [117, 181, 161, 107, 26, 102, 41, 252, 87, 89, 245, 173, 45, 53, 185, 231, 68, 197, 168, 145, 110, 166, 61, 54, 38, 37, 186, 120, 134, 59, 21, 191, 196, 221, 36, 207, 205, 39, 80, 15, 217, 237, 33, 115, 150], [234, 238, 97, 254, 103, 184, 57, 227, 7, 172, 176, 58, 192, 40, 15, 175, 147, 21, 99, 55, 166, 122, 216, 45, 106, 222, 107, 52, 133, 85, 123, 50, 195, 11, 32, 12, 140, 188, 182, 124, 158, 115, 49, 224, 36], [201, 159, 47, 91, 124, 33, 209, 149, 166, 244, 71, 117, 238, 194, 223, 31, 79, 115, 98, 167, 61, 216, 90, 181, 190, 254, 206, 218, 213, 150, 224, 72, 54, 152, 106, 161, 177, 189, 184, 114, 171, 56, 18, 131, 38], [143, 70, 101, 217, 59, 168, 252, 130, 195, 44, 58, 39, 186, 231, 26, 23, 146, 219, 56, 36, 54, 45, 181, 97, 223, 62, 33, 191, 110, 89, 251, 8, 12, 10, 15, 134, 197, 41, 179, 100, 86, 125, 205, 37, 185], [3, 5, 15, 17, 51, 85, 255, 28, 36, 108, 180, 193, 94, 226, 59, 77, 215, 100, 172, 233, 38, 106, 190, 223, 124, 132, 145, 174, 239, 44, 116, 156, 185, 214, 103, 169, 230, 55, 89, 235, 32, 96, 160, 253, 26], [6, 20, 120, 13, 46, 228, 98, 81, 251, 32, 192, 186, 187, 189, 169, 209, 220, 242, 22, 116, 37, 222, 254, 62, 132, 63, 130, 43, 250, 38, 212, 194, 182, 147, 77, 179, 141, 9, 54, 180, 159, 101, 67, 151, 85], [12, 80, 231, 208, 169, 191, 87, 195, 125, 38, 181, 47, 217, 197, 85, 219, 221, 245, 8, 96, 186, 107, 206, 33, 145, 130, 86, 207, 45, 193, 101, 134, 102, 146, 150, 166, 251, 64, 39, 185, 127, 62, 21, 252, 100], [24, 93, 107, 129, 132, 252, 200, 18, 173, 3, 40, 231, 189, 158, 145, 25, 69, 54, 234, 5, 120, 52, 218, 191, 174, 43, 207, 90, 35, 15, 136, 92, 115, 220, 239, 125, 76, 238, 101, 17, 133, 228, 149, 121, 44], [48, 105, 127, 248, 77, 241, 224, 247, 64, 156, 95, 182, 236, 170, 150, 162, 11, 205, 212, 94, 134, 133, 213, 110, 239, 250, 45, 35, 30, 26, 218, 99, 130, 69, 108, 143, 40, 211, 206, 132, 229, 7, 144, 2, 96], [96, 185, 223, 59, 85, 150, 89, 44, 38, 193, 15, 26, 169, 145, 100, 36, 1, 96, 185, 223, 59, 85, 150, 89, 44, 38, 193, 15, 26, 169, 145, 100, 36, 1, 96, 185, 223, 59, 85, 150, 89, 44, 38, 193, 15]], dtype=np.uint16)

print("H_eff.shape =", H_eff.shape)

def get_h_list_for_byte_from_H(H_eff: np.ndarray, byte_idx: int):
    """
    H_eff shape: (m, n_bytes)
    return: coefficient list for one secret byte (one decoder window)
    """
    if H_eff.ndim != 2:
        raise ValueError(f"H_eff must be 2D, got shape={H_eff.shape}")
    m, n_bytes = H_eff.shape
    if not (0 <= byte_idx < n_bytes):
        raise IndexError(f"byte_idx={byte_idx} out of range for n_bytes={n_bytes}")
    return np.array(H_eff[:, byte_idx], dtype=np.uint16)

def run_one_byte_with_real_H(H_eff: np.ndarray, byte_idx: int, rng):
    """
    기존 aggregate_scores_for_one_true_byte(...)를 그대로 사용하되,
    랜덤 h_list 대신 실제 H_eff[:, byte_idx]를 사용한다.
    """
    h_list = get_h_list_for_byte_from_H(H_eff, byte_idx)
    c_true = int(rng.integers(0, 256))
    res = aggregate_scores_for_one_true_byte(c_true, h_list, rng)
    return {
        "byte_idx": int(byte_idx),
        "c_true": int(c_true),
        "h_list": h_list,
        "res": res,
    }


H_eff.shape = (30, 45)


In [11]:
# ----------------------------
# Practical occurrence sampler
# ----------------------------
def sample_one_occurrence_for_true_c_practical(c_true: int, h: int, rng):
    hw_c = hw8_scalar(c_true)
    y_true = gf_mul_ref_python(h, c_true)
    hw_y = hw8_scalar(y_true)

    p_op1 = sample_posterior_by_true_label(op1_hw_pred, op1_hw_buckets, hw_c, rng)
    p_out = sample_posterior_by_true_label(out_hw_pred, out_hw_buckets, hw_y, rng)

    return {
        "c_true": int(c_true),
        "h": int(h),
        "y_true": int(y_true),
        "hw_c": int(hw_c),
        "hw_y": int(hw_y),
        "p_op1": p_op1,
        "p_out": p_out,
    }

def get_h_list_for_byte_from_H(H_eff: np.ndarray, byte_idx: int):
    if H_eff.ndim != 2:
        raise ValueError(f"H_eff must be 2D, got shape={H_eff.shape}")
    m, n_bytes = H_eff.shape
    if not (0 <= byte_idx < n_bytes):
        raise IndexError(f"byte_idx={byte_idx} out of range for n_bytes={n_bytes}")
    return np.array(H_eff[:, byte_idx], dtype=np.uint16)

def aggregate_scores_for_one_true_byte_practical(c_true: int, h_list, rng):
    log_scores = np.zeros(256, dtype=np.float64)
    occ_records = []
    for h in h_list:
        occ = sample_one_occurrence_for_true_c_practical(c_true, int(h), rng)
        occ_records.append(occ)
        for c in range(256):
            hw_c = hw8_scalar(c)
            y = gf_mul_ref_python(int(h), c)
            hw_y = hw8_scalar(y)
            p1 = occ["p_op1"][hw_c]
            p2 = occ["p_out"][hw_y]
            log_scores[c] += np.log(max(p1, 1e-300)) + np.log(max(p2, 1e-300))
    m = np.max(log_scores)
    post = np.exp(log_scores - m)
    post /= post.sum()
    return {
        "post": post,
        "best_c": int(np.argmax(post)),
        "rank_true": posterior_rank(post, c_true),
        "occ_records": occ_records,
    }

def run_one_byte_with_real_H_practical(H_eff: np.ndarray, byte_idx: int, rng):
    h_list = get_h_list_for_byte_from_H(H_eff, byte_idx)
    c_true = int(rng.integers(0, 256))
    res = aggregate_scores_for_one_true_byte_practical(c_true, h_list, rng)
    return {"byte_idx": int(byte_idx), "c_true": int(c_true), "h_list": h_list, "res": res}


## Figure 5 one-window BP

논문 Figure 5의 `gf_mul` 분해를 그대로 따른다.

- `lc = log(c)`
- `s  = lc + log(h)`
- `r  = s mod 255`
- `y  = exp(r)`

그리고 각 occurrence마다
- `hc_i = HW(c)` evidence
- `hy_i = HW(y_i)` evidence
를 넣는다.


In [12]:
# ----------------------------
# Figure 5 tables
# ----------------------------
NC_FIG5 = 512
GF_SENTINEL = 511
GF_ALPHA = 2

def build_hqc_logexp_tables():
    exp_raw = np.zeros(255, dtype=np.uint32)
    log_raw = np.full(256, GF_SENTINEL, dtype=np.uint32)

    x = 1
    for i in range(255):
        exp_raw[i] = x
        log_raw[x] = i
        x = gf_mul_ref_python(GF_ALPHA, x)

    assert x == 1, "GF_ALPHA is not primitive or gf_mul_ref_python mismatch"

    hw512 = np.zeros(NC_FIG5, dtype=np.uint32)
    hw512[:256] = np.array([hw8_scalar(v) for v in range(256)], dtype=np.uint32)

    logz512 = np.zeros(NC_FIG5, dtype=np.uint32)
    logz512[:256] = log_raw

    mod255z = np.zeros(NC_FIG5, dtype=np.uint32)
    for v in range(NC_FIG5):
        if v == GF_SENTINEL:
            mod255z[v] = GF_SENTINEL
        elif v <= 508:
            mod255z[v] = v % 255
        else:
            mod255z[v] = GF_SENTINEL

    expz512 = np.zeros(NC_FIG5, dtype=np.uint32)
    for i in range(255):
        expz512[i] = exp_raw[i]
    expz512[GF_SENTINEL] = 0

    return hw512, logz512, mod255z, expz512, log_raw, exp_raw

def hw_posterior_to_512state(p_hw):
    evid = np.zeros(NC_FIG5, dtype=np.float64)
    evid[:9] = p_hw.astype(np.float64)
    evid /= evid.sum()
    return evid

def byte_uniform_support_512():
    evid = np.zeros(NC_FIG5, dtype=np.float64)
    evid[:256] = 1.0 / 256.0
    return evid

def build_addh_table(h, log_raw):
    h = int(h)
    if h == 0:
        raise ValueError("H coefficient should be non-zero")
    h_log = int(log_raw[h])
    addh = np.zeros(NC_FIG5, dtype=np.uint32)
    for x in range(NC_FIG5):
        if x == GF_SENTINEL:
            addh[x] = GF_SENTINEL
        elif x < 255:
            addh[x] = x + h_log
        else:
            addh[x] = GF_SENTINEL
    return addh

def get_distribution_1d(dist):
    dist = np.asarray(dist, dtype=np.float64)
    if dist.ndim == 2:
        dist = dist[0]
    return dist

hw512, logz512, mod255z, expz512, log_raw, exp_raw = build_hqc_logexp_tables()

# sanity check on one coefficient
h_test = 2
addh_test = build_addh_table(h_test, log_raw)
ok = True
for c in range(256):
    lc = logz512[c]
    s = addh_test[lc]
    r = mod255z[s]
    y = expz512[r]
    if int(y) != int(gf_mul_ref_python(h_test, c)):
        ok = False
        print("Mismatch at c=", c)
        break
print("Figure 5 chain sanity:", ok)


Figure 5 chain sanity: True


In [13]:
# ----------------------------
# Practical Figure 5 one-window BP
# ----------------------------
def run_figure5_window_bp_practical(H_eff, byte_idx, rng):
    trial = run_one_byte_with_real_H_practical(H_eff, byte_idx=byte_idx, rng=rng)
    c_true = trial["c_true"]
    h_list = trial["h_list"]
    manual_res = trial["res"]
    occ_records = manual_res["occ_records"]

    tables = {
        "hw": hw512,
        "logz": logz512,
        "mod255z": mod255z,
        "expz": expz512,
    }

    graph_desc = f"NC {NC_FIG5}\n"
    graph_desc += "TABLE hw\nTABLE logz\nTABLE mod255z\nTABLE expz\n"
    graph_desc += "VAR SINGLE c\n"

    for i, h in enumerate(h_list):
        addh_name = f"addh{i}"
        lc_name, s_name, r_name = f"lc{i}", f"s{i}", f"r{i}"
        y_name, hc_name, hy_name = f"y{i}", f"hc{i}", f"hy{i}"

        tables[addh_name] = build_addh_table(h, log_raw)

        graph_desc += f"TABLE {addh_name}\n"
        graph_desc += f"VAR SINGLE {lc_name}\nVAR SINGLE {s_name}\nVAR SINGLE {r_name}\n"
        graph_desc += f"VAR SINGLE {y_name}\nVAR SINGLE {hc_name}\nVAR SINGLE {hy_name}\n"
        graph_desc += f"PROPERTY {lc_name} = logz[c]\n"
        graph_desc += f"PROPERTY {s_name}  = {addh_name}[{lc_name}]\n"
        graph_desc += f"PROPERTY {r_name}  = mod255z[{s_name}]\n"
        graph_desc += f"PROPERTY {y_name}  = expz[{r_name}]\n"
        graph_desc += f"PROPERTY {hc_name} = hw[c]\n"
        graph_desc += f"PROPERTY {hy_name} = hw[{y_name}]\n"

    factor_graph = FactorGraph(graph_desc, tables)
    bp = BPState(factor_graph, 1, {})
    bp.set_evidence("c", np.ascontiguousarray(byte_uniform_support_512(), dtype=np.float64))

    for i, occ in enumerate(occ_records):
        bp.set_evidence(f"hc{i}", np.ascontiguousarray(hw_posterior_to_512state(occ["p_op1"]), dtype=np.float64))
        bp.set_evidence(f"hy{i}", np.ascontiguousarray(hw_posterior_to_512state(occ["p_out"]), dtype=np.float64))

    bp.bp_acyclic("c")
    bp_dist_512 = get_distribution_1d(bp.get_distribution("c"))
    bp_dist_512 /= bp_dist_512.sum()

    bp_post = bp_dist_512[:256].copy()
    bp_post /= bp_post.sum()

    manual_post = manual_res["post"].astype(np.float64)
    manual_post /= manual_post.sum()

    return {
        "byte_idx": int(byte_idx),
        "c_true": int(c_true),
        "h_list": np.array(h_list, dtype=np.uint16),
        "manual_res": manual_res,
        "bp_post": bp_post,
        "bp_best": int(np.argmax(bp_post)),
        "bp_rank_true": posterior_rank(bp_post, c_true),
        "l1_diff": float(np.sum(np.abs(manual_post - bp_post))),
        "max_abs_diff": float(np.max(np.abs(manual_post - bp_post))),
    }


In [14]:
# ----------------------------
# Practical evaluation
# ----------------------------
byte_idx = 0
window_res = run_figure5_window_bp_practical(H_eff, byte_idx=byte_idx, rng=rng)
print("[practical one-window]")
print("byte_idx      =", byte_idx)
print("true c        =", hex(int(window_res["c_true"])))
print("bp best_c     =", hex(int(window_res["bp_best"])))
print("bp true-rank  =", window_res["bp_rank_true"])
print("L1 diff       =", window_res["l1_diff"])
print("max abs diff  =", window_res["max_abs_diff"])

all_window_results = [run_figure5_window_bp_practical(H_eff, byte_idx=j, rng=rng) for j in range(H_eff.shape[1])]
byte_posteriors = np.stack([res["bp_post"] for res in all_window_results], axis=0)
byte_true = np.array([res["c_true"] for res in all_window_results], dtype=np.uint16)
byte_best = np.array([res["bp_best"] for res in all_window_results], dtype=np.uint16)
byte_ranks = np.array([res["bp_rank_true"] for res in all_window_results], dtype=np.int32)

print("\n[practical all-window summary]")
print("top-1 accuracy =", float(np.mean(byte_best == byte_true)))
print("mean rank      =", float(np.mean(byte_ranks)))
print("median rank    =", float(np.median(byte_ranks)))
for k in [1,2,4,8,16,32,64,128]:
    print(f"top-{k:3d} accuracy = {np.mean(byte_ranks <= k):.4f}")

topk_candidates = np.argsort(byte_posteriors, axis=1)[:, ::-1][:, :TOPK]
topk_scores = np.take_along_axis(byte_posteriors, topk_candidates, axis=1)

attack_package = {
    "H_eff": H_eff,
    "byte_posteriors": byte_posteriors,
    "hard_decision": byte_best.copy(),
    "topk_candidates": topk_candidates,
    "topk_scores": topk_scores,
    "byte_true": byte_true,
    "byte_ranks": byte_ranks,
}
np.savez("rs_decoder_sasca_attack_package_practical.npz", **attack_package)
print("\nSaved: rs_decoder_sasca_attack_package_practical.npz")


[practical one-window]
byte_idx      = 0
true c        = 0xfa
bp best_c     = 0x97
bp true-rank  = 75
L1 diff       = 8.769659874204385e-15
max abs diff  = 2.886579864025407e-15

[practical all-window summary]
top-1 accuracy = 0.0
mean rank      = 35.06666666666667
median rank    = 34.0
top-  1 accuracy = 0.0000
top-  2 accuracy = 0.0000
top-  4 accuracy = 0.0444
top-  8 accuracy = 0.0444
top- 16 accuracy = 0.1778
top- 32 accuracy = 0.4889
top- 64 accuracy = 0.8667
top-128 accuracy = 1.0000

Saved: rs_decoder_sasca_attack_package_practical.npz


## Multi-run practical evaluation + input posterior diagnostics

practical prediction matrix를 여러 번 샘플링해서 전체 공격 통계를 집계하고, 각 `gf_mul` occurrence의 `operand1 HW`와 `output HW` posterior가 실제 정답 HW에 준 확률을 출력한다.

In [15]:
# ============================================================
# Multi-run practical evaluation + posterior diagnostics
# ============================================================
TAU = 19       # HQC128 list-decoding radius used in the paper
TOPK = 128


def entropy(p, eps=1e-300):
    p = np.asarray(p, dtype=np.float64)
    p = p / np.sum(p)
    return float(-np.sum(p * np.log2(np.maximum(p, eps))))


def summarize_occurrence_posteriors(occ_records):
    rows = []
    for idx, occ in enumerate(occ_records):
        p1 = np.asarray(occ["p_op1"], dtype=np.float64)
        p2 = np.asarray(occ["p_out"], dtype=np.float64)
        hw_c = int(occ["hw_c"])
        hw_y = int(occ["hw_y"])
        rows.append({
            "i": idx, "h": int(occ["h"]), "y_true": int(occ["y_true"]),
            "hw_c": hw_c, "hw_y": hw_y,
            "op1_true_prob": float(p1[hw_c]),
            "op1_pred_hw": int(np.argmax(p1)),
            "op1_top_prob": float(np.max(p1)),
            "op1_top1": bool(np.argmax(p1) == hw_c),
            "op1_entropy": entropy(p1),
            "out_true_prob": float(p2[hw_y]),
            "out_pred_hw": int(np.argmax(p2)),
            "out_top_prob": float(np.max(p2)),
            "out_top1": bool(np.argmax(p2) == hw_y),
            "out_entropy": entropy(p2),
        })
    return rows


def print_one_window_input_probabilities(window_res, max_rows=30):
    occ_records = window_res["manual_res"]["occ_records"]
    rows = summarize_occurrence_posteriors(occ_records)
    print("\n[input posterior diagnostics: one window]")
    print("byte_idx =", window_res["byte_idx"], "true c =", hex(int(window_res["c_true"])))
    print("i | h | hw_c | op1_pred p(true) p(max) ok | hw_y | out_pred p(true) p(max) ok")
    for r in rows[:max_rows]:
        print(f"{r['i']:2d} | {r['h']:3d} | {r['hw_c']:4d} | "
              f"{r['op1_pred_hw']:8d} {r['op1_true_prob']:.4f} {r['op1_top_prob']:.4f} {str(r['op1_top1']):5s} | "
              f"{r['hw_y']:4d} | {r['out_pred_hw']:8d} {r['out_true_prob']:.4f} {r['out_top_prob']:.4f} {str(r['out_top1']):5s}")
    op1_true_probs = np.array([r["op1_true_prob"] for r in rows])
    out_true_probs = np.array([r["out_true_prob"] for r in rows])
    op1_top1 = np.array([r["op1_top1"] for r in rows], dtype=bool)
    out_top1 = np.array([r["out_top1"] for r in rows], dtype=bool)
    print("\n[one-window evidence summary]")
    print(f"op1 HW top1 acc      = {op1_top1.mean():.4f}")
    print(f"out HW top1 acc      = {out_top1.mean():.4f}")
    print(f"op1 true prob mean   = {op1_true_probs.mean():.6f}, median={np.median(op1_true_probs):.6f}, min={op1_true_probs.min():.6f}")
    print(f"out true prob mean   = {out_true_probs.mean():.6f}, median={np.median(out_true_probs):.6f}, min={out_true_probs.min():.6f}")


def run_full_window_attack_once_practical_multirun(H_eff, rng):
    all_window_results = [run_figure5_window_bp_practical(H_eff, byte_idx=j, rng=rng) for j in range(H_eff.shape[1])]
    byte_posteriors = np.stack([res["bp_post"] for res in all_window_results], axis=0)
    byte_true = np.array([res["c_true"] for res in all_window_results], dtype=np.uint16)
    byte_best = np.array([res["bp_best"] for res in all_window_results], dtype=np.uint16)
    byte_ranks = np.array([res["bp_rank_true"] for res in all_window_results], dtype=np.int32)
    hard_errors = int(np.sum(byte_best != byte_true))
    op1_ok, out_ok = [], []
    op1_true_probs, out_true_probs = [], []
    op1_entropy, out_entropy = [], []
    for wr in all_window_results:
        for r in summarize_occurrence_posteriors(wr["manual_res"]["occ_records"]):
            op1_ok.append(r["op1_top1"]); out_ok.append(r["out_top1"])
            op1_true_probs.append(r["op1_true_prob"]); out_true_probs.append(r["out_true_prob"])
            op1_entropy.append(r["op1_entropy"]); out_entropy.append(r["out_entropy"])
    return {
        "success_hard": bool(hard_errors == 0),
        "redecode_success_approx": bool(hard_errors <= TAU - 1),
        "hard_errors": hard_errors,
        "top1_acc": float(np.mean(byte_best == byte_true)),
        "mean_rank": float(np.mean(byte_ranks)),
        "median_rank": float(np.median(byte_ranks)),
        "topk_acc": {k: float(np.mean(byte_ranks <= k)) for k in [1,2,4,8,16,32,64,128]},
        "op1_hw_top1": float(np.mean(op1_ok)),
        "out_hw_top1": float(np.mean(out_ok)),
        "op1_true_prob_mean": float(np.mean(op1_true_probs)),
        "out_true_prob_mean": float(np.mean(out_true_probs)),
        "op1_true_prob_median": float(np.median(op1_true_probs)),
        "out_true_prob_median": float(np.median(out_true_probs)),
        "op1_entropy_mean": float(np.mean(op1_entropy)),
        "out_entropy_mean": float(np.mean(out_entropy)),
        "byte_true": byte_true, "byte_best": byte_best, "byte_ranks": byte_ranks,
        "byte_posteriors": byte_posteriors, "all_window_results": all_window_results,
    }


def monte_carlo_practical_multirun(H_eff, n_attacks=20, seed=20260427, print_every=1):
    rng_local = np.random.default_rng(seed)
    results = []
    for t in range(n_attacks):
        res = run_full_window_attack_once_practical_multirun(H_eff, rng_local)
        results.append(res)
        if print_every and ((t + 1) % print_every == 0):
            print(f"trial {t+1:03d}/{n_attacks}: hard_err={res['hard_errors']:2d}, "
                  f"redec~={int(res['redecode_success_approx'])}, byte_top1={res['top1_acc']:.4f}, "
                  f"mean_rank={res['mean_rank']:.2f}, op1_hw_acc={res['op1_hw_top1']:.4f}, "
                  f"out_hw_acc={res['out_hw_top1']:.4f}, op1_ptrue={res['op1_true_prob_mean']:.4f}, "
                  f"out_ptrue={res['out_true_prob_mean']:.4f}")
    def avg(key): return float(np.mean([r[key] for r in results]))
    print("\n[Monte Carlo practical summary]")
    print("n_attacks                 =", n_attacks)
    print("hard success rate          =", avg("success_hard"))
    print("re-decoding success approx =", avg("redecode_success_approx"), "  # hard_errors <= TAU-1")
    print("avg hard errors            =", avg("hard_errors"))
    print("avg byte top-1 acc         =", avg("top1_acc"))
    print("avg mean rank              =", avg("mean_rank"))
    print("avg median rank            =", avg("median_rank"))
    for k in [1,2,4,8,16,32,64,128]:
        print(f"avg top-{k:3d} acc          = {np.mean([r['topk_acc'][k] for r in results]):.4f}")
    print("\n[evidence quality over all sampled gf_mul calls]")
    print("avg op1 HW top-1 acc       =", avg("op1_hw_top1"))
    print("avg out HW top-1 acc       =", avg("out_hw_top1"))
    print("avg op1 P(true HW)         =", avg("op1_true_prob_mean"))
    print("avg out P(true HW)         =", avg("out_true_prob_mean"))
    print("avg op1 entropy            =", avg("op1_entropy_mean"))
    print("avg out entropy            =", avg("out_entropy_mean"))
    return results


# Example 1: inspect one byte in detail
_dbg_rng = np.random.default_rng(12345)
_dbg_window = run_figure5_window_bp_practical(H_eff, byte_idx=0, rng=_dbg_rng)
print("[debug one-window result]")
print("true c       =", hex(int(_dbg_window["c_true"])))
print("bp best      =", hex(int(_dbg_window["bp_best"])))
print("true rank    =", _dbg_window["bp_rank_true"])
print("L1 diff      =", _dbg_window["l1_diff"])
print_one_window_input_probabilities(_dbg_window, max_rows=30)

# Example 2: run multiple complete attacks
practical_mc_results = monte_carlo_practical_multirun(H_eff, n_attacks=20, seed=777, print_every=1)

[debug one-window result]
true c       = 0xb2
bp best      = 0x93
true rank    = 56
L1 diff      = 5.661475237736023e-15

[input posterior diagnostics: one window]
byte_idx = 0 true c = 0xb2
i | h | hw_c | op1_pred p(true) p(max) ok | hw_y | out_pred p(true) p(max) ok
 0 |   2 |    4 |        4 0.5965 0.5965 True  |    5 |        4 0.1985 0.2832 False
 1 |   4 |    4 |        4 0.4807 0.4807 True  |    5 |        4 0.1802 0.2569 False
 2 |   8 |    4 |        3 0.2847 0.5941 False |    6 |        5 0.0796 0.2927 False
 3 |  16 |    4 |        5 0.3130 0.4856 False |    7 |        4 0.0414 0.2800 False
 4 |  32 |    4 |        4 0.5944 0.5944 True  |    4 |        4 0.2755 0.2755 True 
 5 |  64 |    4 |        5 0.4180 0.4469 False |    5 |        3 0.1848 0.2709 False
 6 | 128 |    4 |        5 0.0389 0.4414 False |    4 |        4 0.2749 0.2749 True 
 7 |  29 |    4 |        4 0.5766 0.5766 True  |    4 |        4 0.2654 0.2654 True 
 8 |  58 |    4 |        4 0.5663 0.5663 True  |   

In [16]:
import numpy as np

def analyze_hw_posterior(P, y, name):
    pred = P.argmax(axis=1)
    acc = np.mean(pred == y)

    print(f"\n[{name}]")
    print("top-1 acc =", acc)
    print("mean P(true) =", np.mean(P[np.arange(len(y)), y]))
    print("median P(true) =", np.median(P[np.arange(len(y)), y]))

    # confusion matrix
    C = np.zeros((9, 9), dtype=int)
    for t, p in zip(y, pred):
        C[int(t), int(p)] += 1

    print("\nconfusion matrix: rows=true HW, cols=pred HW")
    print(C)

    # distance별 평균 확률
    print("\nmean posterior by |candidate_hw - true_hw|")
    for d in range(9):
        vals = []
        for i in range(len(y)):
            t = int(y[i])
            for k in range(9):
                if abs(k - t) == d:
                    vals.append(P[i, k])
        if vals:
            print(f"d={d}: mean={np.mean(vals):.6f}")

    # prediction bias
    print("\npredicted class histogram")
    print(np.bincount(pred, minlength=9) / len(pred))

    print("\ntrue class histogram")
    print(np.bincount(y, minlength=9) / len(y))

In [17]:
with h5py.File(PRED_PATH, "r") as f:
    P_op1 = f["op1_hw"]["pred_matrix"][:]
    y_op1 = f["op1_hw"]["true_labels"][:]

    P_out = f["out_hw"]["pred_matrix"][:]
    y_out = f["out_hw"]["true_labels"][:]

analyze_hw_posterior(P_op1, y_op1, "operand1 HW")
analyze_hw_posterior(P_out, y_out, "output HW")


[operand1 HW]
top-1 acc = 0.53946
mean P(true) = 0.41287273
median P(true) = 0.44775277

confusion matrix: rows=true HW, cols=pred HW
[[  340    79     0     0     0     0     0     0     0]
 [   35  1586  1418    70     3     0     0     1     0]
 [   19   529  5925  4157   313    12     1     1     0]
 [    4    92  2263 12668  6411   443    16     3     0]
 [    1    25   282  5120 16123  5342   324    22     4]
 [    0     0    33   638  6772 12024  2344   115     7]
 [    0     0     2    64   711  4921  4484   571    25]
 [    0     0     0     9    32   529  1861   754    68]
 [    0     0     0     0     2    13   139   208    42]]

mean posterior by |candidate_hw - true_hw|
d=0: mean=0.412873
d=1: mean=0.237652
d=2: mean=0.053257
d=3: mean=0.006063
d=4: mean=0.000529
d=5: mean=0.000058
d=6: mean=0.000021
d=7: mean=0.000035
d=8: mean=0.000000

predicted class histogram
[0.00399 0.02311 0.09923 0.22726 0.30367 0.23284 0.09169 0.01675 0.00146]

true class histogram
[0.00419 0.03